# Assignment


## Brief

Write the Python codes for the following questions.


## Instructions

- Step 1: Run the followings cell to get connected to your MongoDB. Make sure your credential is in dotenv file.
- Step 2: Put your answer inside each function. Please do not construct your own function.
- Step 3: You can test your function under test section.
- Step 4. Run test my function to confirm if my code is working.


### Connections

In [1]:
import os

from dotenv import load_dotenv
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi
import pandas as pd
import numpy as np

print("All imported")

All imported


In [2]:
# Load environment variables from .env file
load_dotenv()
MONGODB_URI = os.getenv("MONGODB_URI")
if not MONGODB_URI:
    raise ValueError(
        "❌ MONGODB_URI not found!\n"
        "Please create a .env file with your MongoDB credentials.\n"
        "See README.md for setup instructions."
    )
client = MongoClient(MONGODB_URI, server_api=ServerApi("1"))
# Send a ping to confirm a successful connection
try:
    client.admin.command("ping")
    print("✅ Successfully connected to MongoDB!")
except Exception as e:
    print(e)

✅ Successfully connected to MongoDB!


In [3]:
db = client.sample_mflix
movies = db.movies
print(f"📊 Database: {db.name}")
print(f"📁 Collection: movies ({movies.count_documents({})} documents)")

📊 Database: sample_mflix
📁 Collection: movies (21352 documents)



### Question 1

Question: From the `movies` collection, return the documents with the `plot` that starts with `"war"` in acending order of released date, print only title, plot and released fields. Limit the result to 5.


**Answer:**

> Uses `find()` with `$regex` (like SQL `WHERE plot LIKE 'war%'`), chained with `.sort()` ascending and `.limit(5)` — same pattern as the lesson's spy-movie and "Once upon a time" examples.


In [4]:
import pymongo

for m in movies.find(
    {"plot": {"$regex": "^war", "$options":"i"}}
    ).sort('released', pymongo.ASCENDING).limit(10):

    print(f"Title: {m['title']}\nPlot:{m['plot']}.\nReleased in {m['released']}\n\n")


Title: Nausicaè of the Valley of the Wind
Plot:Warrior/pacifist Princess Nausicaè desperately struggles to prevent two warring nations from destroying themselves and their dying planet..
Released in 1984-03-11 00:00:00


Title: Nausicaè of the Valley of the Wind
Plot:Warrior/pacifist Princess Nausicaè desperately struggles to prevent two warring nations from destroying themselves and their dying planet..
Released in 1984-03-11 00:00:00


Title: Heaven and Earth
Plot:Warlords Kagetora and Takeda each wish to prevent the other from gaining hegemony in feudal Japan. The two samurai leaders pursue one another across the countryside, engaging in massive ....
Released in 1991-02-08 00:00:00


Title: Under the Stars
Plot:Warning! This synopsis contains spoilers Bajo las estrellas (beneath the stars) features the selfish....
Released in 2007-06-15 00:00:00


Title: Aliens vs. Predator: Requiem
Plot:Warring alien and predator races descend on a small town, where unsuspecting residents must band

### Question 2

Question: Group by `rated` and count the number of movies in each.

**Answer:**

> `$group` is the MongoDB equivalent of SQL `GROUP BY rated, COUNT(*)`. The `$sum: 1` accumulator counts one document per group.


In [5]:
group_rated = {
   "$group": { # grp movies by the "rated" category
         "_id": "$rated",
         # count the number of movies in each category, for loop to add 1
         "Movie count": { "$sum": 1 }, 
   }
}

pipeline = [group_rated]
results = movies.aggregate(pipeline)
for rating_summary in results:
   print(rating_summary)


{'_id': 'Approved', 'Movie count': 5}
{'_id': 'M', 'Movie count': 37}
{'_id': 'AO', 'Movie count': 3}
{'_id': 'TV-Y7', 'Movie count': 3}
{'_id': 'TV-G', 'Movie count': 59}
{'_id': 'TV-MA', 'Movie count': 60}
{'_id': 'TV-14', 'Movie count': 89}
{'_id': 'PG-13', 'Movie count': 2321}
{'_id': 'Not Rated', 'Movie count': 1}
{'_id': 'OPEN', 'Movie count': 1}
{'_id': 'R', 'Movie count': 5537}
{'_id': 'PASSED', 'Movie count': 181}
{'_id': 'PG', 'Movie count': 1852}
{'_id': None, 'Movie count': 9897}
{'_id': 'TV-PG', 'Movie count': 76}
{'_id': 'GP', 'Movie count': 44}
{'_id': 'G', 'Movie count': 477}
{'_id': 'APPROVED', 'Movie count': 709}



### Question 3

Question: Count the number of movies with 3 comments or more.


**Answer:**


> Builds on the lesson's inner-join example: `$lookup` joins comments, `$addFields` computes a count per movie, `$match` filters to ≥3, and `$count` returns the total number of matching movies.


In [ ]:
lookup_comments = {
    "$lookup": {
        "from": "comments",
        "localField": "_id",
        "foreignField": "movie_id",
        "as": "comments"
    }
}

add_comment_count = {
    "$addFields": {
        "comment_count": {"$size": "$comments"}
    }
}

match_comments = {
    "$match": {
        "comment_count": {"$gte": 3}
    }
}

count_movies = {
    "$count": "movies_gte_3"
}

pipeline = [
    lookup_comments,
    add_comment_count,
    match_comments,
    count_movies
]

result = list(movies.aggregate(pipeline))
print(result)


In [ ]:
# alternative 
comments = db.comments

pipeline = [
    {"$group": {"_id": "$movie_id", "comment_count": {"$sum": 1}}},
    {"$match": {"comment_count": {"$gte": 3}}},
    {"$count": "movies_gte_3"}
]

result = list(comments.aggregate(pipeline))
print(result)


[{'movies_gte_3': 400}]
